# SIAM Conference on Computational Science and Engineering
`CSE19` https://meetings.siam.org/program.cfm?CONFCODE=CS19<br>
`CSE23` https://meetings.siam.org/program.cfm?CONFCODE=cse23<br>
`CSE25` https://meetings.siam.org/program.cfm?CONFCODE=cse25<br>

## Initialization

In [1]:
# ==== Step 0. Install packages ====
# Install the pyalex library, which provides a Python interface to the OpenAlex API.
!pip install pyalex
# Install the pycountry library, which provides access to ISO 3166 country data.
!pip install pycountry
# Install the country_converter library, which helps in converting country names and codes between different formats.
!pip install country_converter

In [2]:
# ==== Step 0. Import libraries ====
import json
import datetime as dt
import pyalex
import pandas as pd
import country_converter as coco
import re
import pycountry
import pyalex
import yaml

from pyalex import Authors,Institutions, autocomplete, config
from concurrent.futures import ThreadPoolExecutor, as_completed
from pprint import pprint
from tqdm import tqdm

In [3]:
# ==== Step 0. Set differnt conference dataset ====
CONFIG = {
    "CSE19": {
        "file": "CSE19_schd.json",
        "citation_years": [2015,2016,2017,2018,2019]
    },
    "CSE23": {
        "file": "CSE23_schd.json",
        "citation_years": [2019,2020,2021,2022,2023]
    },
    "CSE25": {
        "file": "CSE25_schd.json",
        "citation_years": [2021,2022,2023,2024,2025]
    }
}

# Choose which conference to run
which = "CSE25"  # CSE19, CSE23, CSE25
RESULT_PREFIX = f"{which}_"


INPUT_FILE = CONFIG[which]["file"]
citation_years = CONFIG[which]["citation_years"]

this_year = dt.date.today().year
cutoff = this_year - 5  # five full calendar years back (e.g. 2020 if today is 2025)

# ==== Step 0. Load JSON File ====
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Processing {which}: citation_years = {citation_years}")

Processing CSE25: citation_years = [2021, 2022, 2023, 2024, 2025]


In [4]:
# ==== Step 0. Configure pyalex ====
# Set your email address for the OpenAlex API. This is good practice.
pyalex.config.email = "nana0340@gmail.com"
# Set the maximum number of retries for API requests in case of failures.
config.max_retries = 5
# Set the backoff factor for retries, which determines the waiting time between retries.
config.retry_backoff_factor = 0.1
# Set the HTTP status codes that should trigger a retry.
config.retry_http_codes = [429, 500, 503]

## Functions

In [5]:
# ==== Stage 1. Data Cleaning - Manual Fixing ====
# Fix some specific manual data entry mistakes based on affiliation or email

# CSE19
def cleanCSE19(author_list):
  for a in author_list:

    if a["email"] == "riakymch@cs.umu.se":
      a["affiliation"] = "KTH Royal Institute of Technology, Sweden"
    # if a["email"] == "fbalboausabiaga@flatironinstitute.org":
    # a["affiliation"] = "Flatiron Institute, U.S."

    if a["email"] == "rzamoraresendiz@lbl.gov":
      a["affiliation"] = "Lawrence Berkeley National Laboratory, U.S."

    if a["email"] == "sjin@wisc.edu":
      a["affiliation"] = "University of Wisconsin, U.S."

    if a["email"] == "nmundis@miner.mst.edu":
      a["affiliation"] = "Air Force Research Labratory, U.S."

    if a["affiliation"] == "Simons Foundation and Flatiron Institute, U.S.":
      a["affiliation"] = "Flatiron Institute, U.S."

    if a["email"] == "George.Turkiyyah@kaust.edu.sa":
      a["affiliation"] = "King Abdullah University of Science & Technology, Saudi Arabia"

    if a["email"] == "linlin@math.berkeley.edu":
      a["affiliation"] = "Lawrence Berkeley National Laboratory, U.S."

    if a["email"] == "swartz10@llnl.gov":
      a["affiliation"] = "Lawrence Livermore National Laboratory, U.S."

    if a["email"] == "schoi32@iit.edu":
      a["affiliation"] = "Lawrence Livermore National Laboratory, U.S."

    if a["email"] == "kawahara@imi.kyushu-u.ac.jp":
      a["affiliation"] = "Osaka University, Japan"

    if a["email"] == "sebastien.riffaud@inria.fr":
      a["affiliation"] = "University of Bordeaux, France"

    if a["email"] == "jryan@kth.se":
      a["affiliation"] = "University of East Anglia, United Kingdom"

    if a["email"] == "matrw@nus.edu.sg":
      a["affiliation"] = "National University of Singapore, Singapore"

    if a["email"] =="fractor@eecs.berkeley.edu":
      a["affiliation"] = "Lawrence Livermore National Laboratory, U.S."

    if a["email"] == "kklymko@lbl.gov":
      a["affiliation"] = "Lawrence Livermore National Laboratory, U.S."

    if a["email"] == "guillaume.delay@enpc.fr":
      a["affiliation"] = "Lawrence Livermore National Laboratory, U.S."

    if a["email"] == "dongarra@icl.utk.edu":
      a["affiliation"] = "Oak Ridge National Laboratory, U.S."

    if a["email"] == "sheroze@ices.utexas.edu":
      a["affiliation"] = "The University of Texas at Austin, U.S."

    if a["email"] == "olegbalabanov@gmail.com":
      a["affiliation"] = "Polytechnic University of Catalonia, Spain"

    if a["email"] == "jung153@postech.ac.kr":
      a["affiliation"] = "Ajou University, Korea"

    if a["email"] == "peter.vanleeuwen@colostate.edu":
      a["affiliation"] = "Colorado State University, U.S."

    if a["email"] == "wxm.math@outlook.com":
      a["affiliation"] = "Florida State University, U.S."

    if a["email"] == "liweiming@csrc.ac.cn":
      a["affiliation"] = "Beijing Computational Science Research Center, China"

    if a["email"] == "dcdelrey@gmail.com":
      a["affiliation"] = "National Institute of Aerospace, U.S."

    if a["email"] == "qwang@math.sc.edu":
      a["affiliation"] = "University of South Carolina, U.S."

    if a["email"] == "maday@ann.jussieu.fr":
      a["affiliation"] = "Brown University, U.S."

    if a["email"] == "wxm.math@outlook.com":
      a["affiliation"] = "Florida State University, U.S."

    if a["affiliation"] == "IT University of Copenhagen, Denmark, Zhoulai@ucdavis":
      a["affiliation"] = "IT University of Copenhagen, Denmark"
      a["email"] = "zhoulai@ucdavis.edu"

    if a["affiliation"] == "Virginia Tech, U.S., ":
      a["affiliation"] = "Virginia Tech, U.S."

    if a["affiliation"] == "Jülich Supercomputing Centre, Germany":
      a["affiliation"] = "Jülich Research Centre, Germany"

    if a["affiliation"] == "Universitat Hamburg, Germany":
      a["affiliation"] = "University of Hamburg, Germany"

    if a["affiliation"] == "University of Illinois at Urbana-Champaign and Lawrence Livermore National Laboratory, U.S.":
      a["affiliation"] = "University of Illinois Urbana-Champaign, U.S."

    if a["affiliation"] == "University of Illinois at Urbana-Champaign, U.S.":
      a["affiliation"] = "University of Illinois Urbana-Champaign, U.S."

    if a["affiliation"] == "NASA Langley Research Center, U.S.":
      a["affiliation"] = "Langley Research Center, U.S."

    if a["affiliation"] == "Technische Universitaet Wien, Austria":
      a["affiliation"] = "TU Wien, Austria"

    if a["affiliation"] == "IBM T.J. Watson Research Center, U.S.":
      a["affiliation"] = "IBM Research - Thomas J. Watson Research Center, U.S."

    if a["affiliation"] == "Courant Institute of Mathematical Sciences, New York University, U.S.":
      a["affiliation"] = "New York University, U.S."

    if a["affiliation"] == "University of Toronto Institute for Aerospace Studies, Canada":
      a["affiliation"] = "University of Toronto, Canada"

    if a["affiliation"] == "University of Goettingen, Germany":
      a["affiliation"] = "University of Göttingen, Germany"

    if a["affiliation"] == "Humboldt University Berlin, Germany":
      a["affiliation"] = "Humboldt-Universität zu Berlin, Germany"

# CSE23
def cleanCSE23(author_list):
  for a in author_list:

    if (a.get("affiliation") == "University of Illinois Urbana-Champaign"):
      a["affiliation"] = "University of Illinois Urbana-Champaign, U.S."

    if (a["email"] == "y.hao@uva.nl"):
      a["affiliation"] = "University of Amsterdam, Netherlands"

    if (a["email"] == "baierreinio@maths.ox.ac.uk"):
      a["affiliation"] = "University of Oxford, United Kingdom"

    if (a["email"] == "sthekke@mpi-cbg.de"):
      a["name"] = "Sachin Krishnan Thekke Veettil"
      a["affiliation"] = "Max Planck Institute of Molecular Cell Biology and Genetics, Germany"

    if (a["email"] == "marien-lorenzo.hanot@univ-lille.fr"):
      a["affiliation"] = "Université Lille 1 and CNRS, France"

    if (a.get("affiliation") == "Katholieke Universiteit Leuven, Belgium, mohamedamine.benyahmed@kuleuven, be"):
      a["affiliation"] = "Katholieke Universiteit Leuven, Belgium."
      a["email"] = "mohamedamine.benyahmed@kuleuven.be"

    if (a.get("affiliation") == "Stanford University, U.S., jb0@stanford"):
      a["affiliation"] = "Stanford University, U.S."
      a["email"] = "jb0@stanford.edu"

    if (a["email"] == "asbhalla@sdsu.edu"):
      a["affiliation"] = "San Diego State University, U.S."

    if (a.get("affiliation") == "Martin Luther University Halle-Wittenberg Germany"):
      a["affiliation"] = "Martin Luther University Halle-Wittenberg, Germany"

    if (a.get("affiliation") == "Universita degli Studi di Palermo - Italia"):
      a["affiliation"] = "Universita degli Studi di Palermo, Italy"

    if (a.get("affiliation") == "University of Alabama, U.S"):
      a["affiliation"] = "University of Alabama, U.S."

    if (a.get("affiliation") == "Indian Institute of Technology (Banaras Hindu University), India"):
      a["affiliation"] = "Indian Institute of Technology BHU, India"

    if (a.get("affiliation") == "King Abdullah University of Science & Technology (KAUST), Saudi Arabia"):
      a["affiliation"] = "King Abdullah University of Science & Technology, Saudi Arabia"

    if a["email"] == "omlin@cscs.ch":
      a["affiliation"] = "ETH Zurich, Switzerland"

    if a["email"] == "omlin@cscs.ch":
      a["affiliation"] = "Eindhoven University of Technology, Netherlands"

    if a["email"] == "konstantinos.ritos@strath.ac.uk":
      a["affiliation"] = "University of Strathclyde, United Kingdom"

    if a["email"] == "francois.pellegrini@u-bordeaux.fr":
      a["affiliation"] = "University of Bordeaux, France"

    if a["email"] == "l.szpruch@ed.ac.uk":
      a["affiliation"] = "University of Edinburgh, United Kingdom"

    if a["email"] == "obc@zurich.ibm.com":
      a["affiliation"] = "ETH Zurich, Switzerland"

    if a["email"] == "carla@simula.no":
      a["affiliation"] = "Simula Metropolitan Center for Digital Engineering, Norway"

    if a["email"] == "eki.agouzal@inria.fr":
      a["affiliation"] = "Centre National de la Recherche Scientifique, France"

    if a["email"] == "ayaboe.edoh@jacobs.com":
      a["affiliation"] = "United States Air Force Research Laboratory, U.S."

    if a["email"] == "nikolaj.mucke@cwi.nl":
      a["affiliation"] = "Centrum Wiskunde & Informatica, Netherlands"

    if a["email"] == "zhoge@mech.kth.se":
      a["affiliation"] = "KTH Royal Institute of Technology, Sweden"

    if a["email"] == "Daan.Crommelin@cwi.nl":
      a["affiliation"] = "Centrum Wiskunde & Informatica, Netherlands"

    if a["email"] == "ccarvalho3@ucmerced.edu":
      a["affiliation"] = "University of California, Merced, U.S."

    if a["email"] == "z.ziani@numeryx.fr":
      a["affiliation"] = "University of Paris-Saclay, France"

    if a["email"] == "accounts@chrisrackauckas.com":
      a["affiliation"] = "MIT, U.S."

    if a["email"] == "eric.sonnendruecker@ipp.mpg.de":
      a["affiliation"] = "Technische Universität München, Germany"

    if a["email"] == "beatrice.battisti@inria.fr":
      a["affiliation"] = "Politecnico di Torino, Italy"

    if a["email"] == "david.keyes@kaust.edu.sa":
      a["affiliation"] = "King Abdullah University of Science & Technology, Saudi Arabia"

    if a["email"] == "a.p.zwart@tue.nl":
      a["affiliation"] = "Eindhoven University of Technology, Netherlands"

    if a["email"] == "bas.van.der.linden@sioux.eu":
      a["affiliation"] = "Technical University of Eindhoven, Netherlands"

    if a["email"] == "maherou@sandia.gov":
      a["affiliation"] = "Sandia National Laboratories, U.S."

    if a["email"] == "richefort.clement@protonmail.com":
      a["affiliation"] = "Lawrence Livermore National Laboratory, U.S."

    if a["email"] == "brian.bantsoukissa@ens-lyon.fr":
      a["affiliation"] = "ENS Lyon, France"

    if a["email"] == "mdolores.gomez@usc.es":
      a["affiliation"] = "Universidade de Santiago de Compostela, Spain"

    if (a.get("affiliation") == "University of Hasselt, Belgium"):
      a["affiliation"] = "Hasselt University, Belgium"

    if (a.get("affiliation") == "German Aerospace Center (DLR), Germany"):
      a["affiliation"] = "German Aerospace Center, Germany"

    if (a.get("affiliation") == "Research Centre Juelich, Germany"):
      a["affiliation"] = "Jülich Research Centre, Germany"

    if (a.get("affiliation") == "Juelich Aachen Research Alliance, Forschungszentrum Juelich, Germany"):
      a["affiliation"] = "Jülich Research Centre, Germany"

    if (a.get("affiliation") == "Centrum voor Wiskunde en Informatica (CWI), Netherlands"):
      a["affiliation"] = "Centrum Wiskunde & Informatica, Netherlands"

    if (a.get("affiliation") == "Universita' di Catania, Italy"):
      a["affiliation"] = "University of Catania, Italy"

# CSE25
def cleanCSE25(author_list):
  for a in author_list:

     if a["email"] == "p.khurana22@imperial.ac.uk":
      a["affiliation"] = "Imperial College London, U.K."

     if a["email"] == "orly@sci.utah.edu":
      a["affiliation"] = "University of Utah, U.S."
     if a["email"] == "pxrodriguez@gwu.edu":
      a["affiliation"] = "George Washington University, U.S."
     if a["email"] == "pbedeka1@jhu.edu":
      a["affiliation"] = "Johns Hopkins University, U.S."
     if a["email"] == "yiping.lu@northwestern.edu":
      a["affiliation"] = "Northwestern University, U.S."

     if a["email"] == "frank.jenko@ipp.mpg.de":
      a["affiliation"] = "University of Texas at Austin, U.S."

     if a["affiliation"] == "Université Lille 1 and CNRS, France":
      a["affiliation"] = "Université de Lille, France"

     if a["email"] == "k.kirchner@tudelft.nl":
      a["affiliation"] = "KTH Royal Institute of Technology, Sweden"

     if a["email"] == "bjoern.sailer@uni-trier.de":
      a["affiliation"] = "University of Linz, Austria"

     if a["email"] == "davidzan830@gmail.com":
      a["affiliation"] = "National Taiwan University, Taiwan"

     if a["email"] == "afroja.parvin@kuleuven.be":
      a["affiliation"] = "Peking University, China"

     if a["email"] == "lgreengard@flatironinstitute.org":
      a["affiliation"] = "Flatiron Institute, U.S."

     if a["email"] == "brandstetter@ml.jku.at":
      a["affiliation"] = "Johannes Kepler University Linz, Austria"

     if a["email"] == "gg434@cornell.edu":
      a["affiliation"] = "Cornell University, U.S."

     if a["email"] == "laura.grigori@epfl.ch":
      a["affiliation"] = "EPFL, Switzerland"

     if a["email"] == "william.kirby@ibm.com":
      a["affiliation"] = "Tufts University, U.S."

     if a["affiliation"] == "Environment Canada, Canada, stephane":
      a["affiliation"] = "Environment Canada, Canada"

     if a["affiliation"] == "Humboldt University Berlin, Germany":
      a["affiliation"] = "Humboldt-Universität zu Berlin, Germany"

     if a["affiliation"] == "Humboldt University at Berlin, Germany":
      a["affiliation"] = "Humboldt-Universität zu Berlin, Germany"

     if a["affiliation"] == "Flatiron Institute, New York University, U.S.":
      a["affiliation"] = "Flatiron Institute, U.S."

     if a["affiliation"] == "Fraunhofer Institut ITWM, Kaiserslautern, Germany":
      a["affiliation"] = "Fraunhofer Institute for Industrial Mathematics, Germany"

     if a["affiliation"] == "RIKEN Advanced Institute for Computational Science, Japan":
      a["affiliation"] = "RIKEN Center for Computational Science, Japan"

     if a["affiliation"] == "Heinrich-Heine Universitaet Duesseldorf, Germany":
      a["affiliation"] = "Heinrich Heine University Düsseldorf, Germany"

     if a["affiliation"] == "Loyola University of Chicago, U.S.":
      a["affiliation"] = "Loyola University Chicago, U.S."

     if a["affiliation"] == "Universita degli Studi di Palermo, Italy":
      a["affiliation"] = "University of Palermo, Italy"

     if a["affiliation"] == "Max Planck Institute, Magdeburg, Germany":
      a["affiliation"] = "Max Planck Institute for Dynamics of Complex Technical Systems, Germany"

     if a["affiliation"] == "Heidelberg University & Bosch Research, Germany":
      a["affiliation"] = "Heidelberg University, Germany"

     if a["affiliation"] == "Universidad Politecnica de Valencia, Spain":
      a["affiliation"] = "Universitat Politècnica de València, Spain"

     if a["affiliation"] == "Universita di Padova, Italy":
      a["affiliation"] = "University of Padua, Italy"

     if a["affiliation"] == "Polytechnique Montreal, Canada":
      a["affiliation"] = "Polytechnique Montréal, Canada"

     if a["affiliation"] == "University of Erlangen-Nuernberg, Germany":
      a["affiliation"] = "University of Erlangen-Nuremberg, Germany"

     if a["affiliation"] == "University of Ilinois at Chicago, U.S.":
      a["affiliation"] = "University of Illinois at Chicago, U.S."

In [6]:
# ==== Stage 2. Convert a country name into its standard ISO 3166-1 alpha-2 code (a two-letter abbreviation) ====

# This function to_iso2 uses the pycountry library to convert
# a given country name to its ISO 3166-1 alpha-2 code.
# It uses a fuzzy search to handle potential variations in the input name.
def to_iso2(name):
    try:
        return pycountry.countries.search_fuzzy(name)[0].alpha_2
    except LookupError:
        return None

In [7]:
# ==== Stage 2. Author search and disambiguation ====
# Regex pattern to remove middle name initial followed by a period
pattern_remove_middle_name_initial = r"(\b[^\W\d_]+(?:[\"-][^\W\d_]+)*)\s+[A-Z]\.\s+([^\W\d_]+(?:[\"-][^\W\d_]+)*)"

def checkOpenAlexData(author_input, debug_print = False):
  # Extract institution name and country from the input author data
  if (author_input["institution"]):
    name_part = author_input["institution"].strip()
  else:
    name_part = None

  if (author_input["country"]):
    country_raw = author_input["country"].strip()
    country_iso = coco.convert(names=country_raw, to="ISO2")
  else:
    country_raw = None
    country_iso = None

  inst_match_method = None
  match_method = None

  # Convert country name to ISO2 code using country_converter
  # Search for institution in OpenAlex, first by name and country, then by name only
  if (country_iso != None):
    r_inst = Institutions().search(name_part).filter(country_code=country_iso).get()
    inst_match_method = "IN_C_N"
  else:
    r_inst = None

  if (r_inst):
    if (debug_print): print("IN_C_N: Y", r_inst)
  else:
    r_inst = Institutions().search(name_part).get()
    if (r_inst):
      inst_match_method = "IN_N"
      if (debug_print): print("IN_N: Y", r_inst)
    else:
      if (debug_print): print("IN_NO_MATCH: N")
      inst_match_method = "IN_NO_MATCH"

  if (debug_print):
    print("Institution :",r_inst)

  # print("====")

  # Search for author in OpenAlex
  if (r_inst):
    # print(r_inst[0])
    # Search by display name and last known institution ID
    r_author = Authors().search_filter(display_name=author_input["name"]).filter(last_known_institutions={"id": r_inst[0]["id"]}).get()
    if (r_author):
      match_method = "I_C_N"
    else:
      # Search by display name and affiliation institution ID
      r_author = Authors().search_filter(display_name=author_input["name"]).filter(affiliations={"institution":{"id": r_inst[0]["id"]}}).get()
      if (r_author):
       match_method = "I_P_N"
      else:
        # Search by display name only
        r_author = Authors().search_filter(display_name=author_input["name"]).get()
        if(r_author):
          match_method = "I_N_N"
        else:
          # Search by name only
          r_author = Authors().search(author_input["name"]).get()
          if (r_author):
            match_method = "I_N_N_A"
          else:
            # Clean name by removing middle initial and search again
            cleaned = re.sub(pattern_remove_middle_name_initial, r"\1 \2", author_input["name"], flags=re.UNICODE)
            r_author = Authors().search_filter(display_name=cleaned).filter(affiliations={"institution":{"id": r_inst[0]["id"]}}).get()
            if (r_author):
              match_method = "I_P_CL_N"
            else:
              match_method = "I_NO_MATCH"
  else:
    # If institution is not found, search for author without institution filter
    print(f"~~~Missing Institution: {author_input["affiliation"]}")
    r_author = Authors().search_filter(display_name=author_input["name"]).get()
    if(r_author):
      match_method = "N"
    else:
      r_author = Authors().search(author_input["name"]).get()
      if (r_author):
        match_method = "N_A"
      else:
        match_method = "NO_MATCH"

  if (debug_print):
    print("Author      :",r_author)

  extracted_data = {}
  author_data = []

  display_name = author_input["name"]

  if r_author:
    # Extract relevant information and flatten the structure

    for author in r_author:
        last_institution = author.get("last_known_institutions")
        if (last_institution):
          last_institution_name = last_institution[0].get("display_name")
        else:
          last_institution_name = None

        summary_stats = author.get("summary_stats")

        # Calculate the number of citations in the last 5 years
        # OpenAlex already bins every author’s citations by year in `counts_by_year`
        recent_cite_count = sum(
            yr["cited_by_count"]
            for yr in author["counts_by_year"]          # field documented here :contentReference[oaicite:1]{index=1}
            if yr["year"] in citation_years
        )

        display_name = author.get("display_name")

        extracted_data = {
            "display_name": author.get("display_name"),
            "display_name_alternatives": author.get("display_name_alternatives"),
            "email": author.get("email"),
            "last_known_institution": last_institution_name,
            "id": author.get("id"),
            "orcid": author.get("orcid"),
            "relevance_score": author.get("relevance_score"),
            "works_count": author.get("works_count"),
            "cited_by_count": author.get("cited_by_count"),
            "h_index": summary_stats.get("h_index","0"),
            "i10_index": summary_stats.get("i10_index","0"),
            "2yr_mean_citedness": summary_stats.get("2yr_mean_citedness","0"),
            "recent_cite_count": recent_cite_count,
        }
        author_data.append(extracted_data)

    # Create a pandas DataFrame and display it if debug_print is True
    if (debug_print):
      df = pd.DataFrame(author_data)
      display(df)
  else:
    if (debug_print):
      print(f"No results found for author: {author_input["name"]}")

  # Select the first result if multiple authors are found (this might need refinement based on specific criteria)
  extracted_data_select = {}
  if (len(author_data)> 0):
    extracted_data_select = author_data[0]

  # Prepare the output data dictionary
  output_data = {
    "input_name":  author_input["name"],
    "email":  author_input["email"],
    "institution":  author_input["institution"],
    "country":  author_input["country"],
    "display_name": display_name,
    "affiliation": author_input["affiliation"],
    "name_part": name_part,
    "country_raw": country_raw,
    "country_iso": country_iso,
    "extracted_data": extracted_data_select,
    "r_author": r_author,
    "r_inst": r_inst,
    "match_method": match_method,
    "inst_match_method": inst_match_method
  }
  return output_data

## Stage 1: Data Preparation and Cleaning

In [8]:
# ==== Step 1. Initialize variables ====
# Initialize a set to store unique authors and a counter for duplicates
author_set = set()
duplicate_counter = 0

# Flag to process only the speaker or all authors in a talk
ONLY_SPEAKER = True

# ==== Step 2. Iterate through minisymposia and talks ====
# Iterate through each minisymposium (ms_key) in the data
for ms_key in data:
  # Get the data for the current minisymposium
  ms = data[ms_key]
  if ms_key.startswith("MS"):
    # Check if the minisymposium has talks
    if (ms.get("talks")):
      # Iterate through each talk in the current minisymposium
      for t in ms["talks"]:
        # ==== # Step 3. Process authors ====
        # If ONLY_SPEAKER is True, process only the speaker
        if (ONLY_SPEAKER):
          # Get the index of the speaker, default to 0 if not found
          speaker_index = t.get("speaker_index", None)
          if (speaker_index == None):
            print(f"!! No speaker_index: {ms_key}, using first one")
            speaker_index = 0
          # ==== Step 4. Deduplicate authors ====
          # Get the speaker"s data
          a = t["authors"][speaker_index]
          # Convert the author dictionary to a JSON string for consistent hashing
          a_json = json.dumps(a, sort_keys=True)
          # Check if the author is already in the set
          if (a_json in author_set):
            # Increment the duplicate counter if the author is a duplicate
            duplicate_counter = duplicate_counter + 1
            # print("Duplicate author:", a_json) # Commented out duplicate print
          else:
            # Add the unique author to the set
            author_set.add(a_json)
        # If ONLY_SPEAKER is False, process all authors in the talk
        else:
          for a in t["authors"]:
            # print("Author:", a) # Commented out author print
            # Convert the author dictionary to a JSON string for consistent hashing
            a_json = json.dumps(a, sort_keys=True)
            # Check if the author is already in the set
            if (a_json in author_set):
              # Increment the duplicate counter if the author is a duplicate
              duplicate_counter = duplicate_counter + 1
              # print("Duplicate author:", a_json) # Commented out duplicate print
            else:
              # Add the unique author to the set
              author_set.add(a_json)
  else:
    print(f"!! Not a minisymposium: {ms_key}")

# ==== Step 5. Convert set back to list ====
# Convert the set of JSON strings back to a list of dictionaries
author_list = [json.loads(author_str) for author_str in author_set]

# ==== Step 6. Print the number of unique and duplicate authors ====
# Print the number of unique and duplicate authors
print("Unique Authors: ",len(author_set))
print("Duplicate Authors: ",duplicate_counter)
# print(author_set) # Removed unnecessary print
print("author_list:")
print(author_list) # Print the list of unique authors

# ==== Step 7. Manually clean some data entries ====
# Manually clean some data entries
cleanCSE23(author_list)
cleanCSE19(author_list)
cleanCSE25(author_list)

# ==== Step 8. Extract institution and country ====
# Define a regular expression pattern to extract institution and country from affiliation string
rx = re.compile(r"^(?P<institution>.+),\s*(?P<country>[^,]+)\s*$")

# Iterate through the author list to extract institution and country using regex
for a in author_list:
  m = rx.fullmatch(a["affiliation"])
  # print(a, "→", m.groupdict() if m else None) # Commented out debug print
  # Extract country and institution if regex matches, otherwise set to None
  a["country"] = m.group("country") if m else None
  a["institution"] = m.group("institution") if m else a["affiliation"]

# ==== Step 9. Identify missing fields ====
# Check data for missing values in key fields
list_missing_name =[]
list_missing_email =[]
list_missing_affiliation =[]
list_missing_country =[]
list_missing_institution =[]
list_multi_institution =[]

# Populate the lists of missing data
for a in author_list:
  if (a.get("name") == None): # Use .get() for safer access
    list_missing_name.append(a)
  if (a.get("email") == None): # Use .get() for safer access
    list_missing_email.append(a)
  if (a.get("affiliation") == None): # Use .get() for safer access
    list_missing_affiliation.append(a)
  if (a.get("country") == None): # Use .get() for safer access
    list_missing_country.append(a)
  if (a.get("institution") == None): # Use .get() for safer access
    list_missing_institution.append(a)
  if re.search(r"\band\b", a.get("affiliation") or "", flags=re.IGNORECASE):
    list_multi_institution.append(a)

# ==== Step 10. Print the counts and details of missing data ====
# Print the counts and details of missing data
print("list_missing_name", len(list_missing_name), list_missing_name)
print("list_missing_email", len(list_missing_email), list_missing_email)
print("list_missing_affiliation", len(list_missing_affiliation), list_missing_affiliation)
print("list_missing_country", len(list_missing_country), list_missing_country)
print("list_missing_institution", len(list_missing_institution), list_missing_institution)
print("list_multi_institution", len(list_multi_institution), list_multi_institution)

!! Not a minisymposium: IP1
!! No speaker_index: MS4, using first one
!! No speaker_index: MS12, using first one
!! Not a minisymposium: PD1
!! Not a minisymposium: SP1
!! Not a minisymposium: SP2
!! Not a minisymposium: PD5
!! No speaker_index: MS29, using first one
!! Not a minisymposium: SP3
!! Not a minisymposium: SP4
!! Not a minisymposium: SP5
!! Not a minisymposium: IP2
!! No speaker_index: MS83, using first one
!! Not a minisymposium: PD2
!! Not a minisymposium: IP3
!! No speaker_index: MS105, using first one
!! No speaker_index: MS110, using first one
!! No speaker_index: MS113, using first one
!! No speaker_index: MS121, using first one
!! No speaker_index: MS132, using first one
!! Not a minisymposium: IP4
!! No speaker_index: MS145, using first one
!! Not a minisymposium: PD3
!! Not a minisymposium: IP5
!! Not a minisymposium: PD6
!! No speaker_index: MS170, using first one
!! No speaker_index: MS177, using first one
!! No speaker_index: MS198, using first one
!! Not a mini

## Stage 2: Author Matching + Call OpenAlex API

In [9]:
# ==== Step 1. TESTING (optional) ====
# Run sample 0-20 one by one

author_map = dict()
a_list = []

for a in tqdm(author_list[0:20]):
  # print("Run",a)
  try:
    a_out = checkOpenAlexData(a, False)
    a_list.append(a_out)
  except Exception as e:
    print("An exception occurred", a, e)
  # json.load(a)
  # a[]

100%|███████████████████████████████████████████| 20/20 [00:21<00:00,  1.07s/it]


In [10]:
# ==== Step 2. Run all data in parallel ====
author_map = dict()
a_list = []

def process_author(a):
    try:
        return checkOpenAlexData(a, False)
    except Exception as e:
        print("An exception occurred", a, e)
        return None  # or some other placeholder

with ThreadPoolExecutor(max_workers = 3) as executor:
    # Submit all tasks to the executor
    futures = {executor.submit(process_author, a): a for a in author_list}

    # Use tqdm with as_completed for progress bar
    for future in tqdm(as_completed(futures), total = len(futures)):
        result = future.result()
        if result is not None:
            a_list.append(result)

# Wait for all threads to finish
executor.shutdown(wait = True)
print(f"All done - {len(a_list)} / {len(author_list)}" )

  3%|█▎                                       | 39/1278 [00:14<07:16,  2.84it/s]

~~~Missing Institution: Solea Energy, U.S.


  4%|█▋                                       | 52/1278 [00:20<09:35,  2.13it/s]

~~~Missing Institution: Max Planck Institute for Computational Methods in Systems and Control Theory, Germany


  9%|███▌                                    | 114/1278 [00:44<06:51,  2.83it/s]

...

~~~Missing Institution: RTX Technology Research Center, U.S.


 97%|█████████████████████████████████████▋ | 1237/1278 [08:30<00:17,  2.34it/s]

~~~Missing Institution: Sorbonne Universités, France


100%|███████████████████████████████████████| 1278/1278 [08:47<00:00,  2.42it/s]

All done - 1278 / 1278


In [11]:
# ==== Step 3. Enrich Data by additional values ====
for a_out in a_list:
    ed = a_out.get("extracted_data", {})

    a_out["id"] = ed.get("id")
    a_out["orcid"] = ed.get("orcid")
    a_out["relevance_score"] = ed.get("relevance_score")
    a_out["works_count"] = ed.get("works_count")
    a_out["cited_by_count"] = ed.get("cited_by_count")
    a_out["h_index"] = ed.get("h_index")
    a_out["i10_index"] = ed.get("i10_index")
    a_out["2yr_mean_citedness"] = ed.get("2yr_mean_citedness")
    a_out["recent_cite_count"] = ed.get("recent_cite_count")
    a_out["display_name_alternatives"] = ed.get("display_name_alternatives")

In [12]:
# ==== Step 4. Display all results in DataFrame ====
df = pd.DataFrame(a_list)

In [13]:
# ==== Step 5. Save results to files ====
selected_columns = ["input_name", "email", "inst_match_method", "match_method", "display_name", "display_name_alternatives",	"affiliation", "institution", "country", "country_iso", "id", "orcid", "relevance_score", "works_count",	"cited_by_count", "h_index", "i10_index", "2yr_mean_citedness",	"recent_cite_count"]

# Export to CSV
df[selected_columns].to_excel(RESULT_PREFIX + "selected_output.xlsx")
df[selected_columns].to_json(RESULT_PREFIX + "selected_output.json", orient = "records")


# Write to a file
with open(RESULT_PREFIX + "output.json", "w", encoding = "utf-8") as f:
    json.dump(a_list, f, indent = 2, ensure_ascii = False)